# 1 · From neighborhoods to covers — the Grothendieck GNN idea

Almost every graph neural network is built on **one** primitive: the
*neighborhood*. A layer updates each node by aggregating over the nodes one hop
away. Stacking layers grows the receptive field, but the atom of computation is
always "a node and its neighbors."

The **Grothendieck GNN (GkGNN)** framework makes a single structural move:

> replace the neighborhood with a **cover** — a more general family of directed
> subgraphs attached to each node — and translate covers into matrices the same
> way the adjacency matrix encodes neighborhoods.

This notebook builds that idea from the ground up with small, concrete matrices:

1. why ordinary message passing is **bounded by the 1-WL test**,
2. what a **cover** is and the homomorphism `Tr` that turns subgraph composition
   into matrix multiplication,
3. three covers we actually use — the **walk**, **reachability**, and **sieve**
   covers — and the category-theory intuition (sites & sieves) behind the last.

The three demo notebooks (`02`–`04`) then put these covers to work on a
lateral-movement detection task.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # repo root, so `import src...` works
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=2, suppress=True)
plt.rcParams["figure.figsize"] = (7, 4)

## 1.1 · Message passing and the 1-WL ceiling

A message-passing layer computes, for every node $v$,
$$h_v^{(t+1)} = \phi\Big(h_v^{(t)},\ \textstyle\bigoplus_{u \in N(v)} h_u^{(t)}\Big),$$
an update from $v$'s own state and the **multiset** of its neighbors' states.

The **1-dimensional Weisfeiler–Leman (1-WL)** test is the same recursion with
hashes instead of vectors: each node's color is refined by hashing
`(own color, sorted multiset of neighbor colors)`. A foundational result
(Xu et al. 2019; Morris et al. 2019) says **no message-passing GNN can
distinguish two graphs that 1-WL colors identically**.

Here is 1-WL in five lines. Watch what it does to two graphs that are *locally*
identical but *globally* different — one directed 6-cycle vs. two directed
3-cycles (the classic $C_6$ vs. $2\cdot C_3$ counterexample, which is the heart
of demo `02`).

In [2]:
from src.data import _flat, _segmented

def wl_refine(A, rounds=6):
    "1-WL color refinement using out-neighbor color multisets."
    A = (np.asarray(A) > 0).astype(int)
    n = A.shape[0]
    colors = np.zeros(n, dtype=int)            # all nodes start identical
    for _ in range(rounds):
        table, new = {}, np.zeros(n, dtype=int)
        for v in range(n):
            sig = (colors[v], tuple(sorted(colors[u] for u in np.nonzero(A[v])[0])))
            new[v] = table.setdefault(sig, len(table))
        colors = new
    return colors

flat = _flat(6)            # one directed 6-cycle
seg  = _segmented(6, 2)    # two directed 3-cycles

print("flat  (one 6-cycle):    distinct 1-WL colors =", np.unique(wl_refine(flat)).size)
print("segmented (two 3-cycles): distinct 1-WL colors =", np.unique(wl_refine(seg)).size)
print("\n1-WL collapses both to a single color: every node has in-degree 1,")
print("out-degree 1 and the same start color, so refinement never separates them.")

flat  (one 6-cycle):    distinct 1-WL colors = 1
segmented (two 3-cycles): distinct 1-WL colors = 1

1-WL collapses both to a single color: every node has in-degree 1,
out-degree 1 and the same start color, so refinement never separates them.


Both graphs collapse to **one** color, so any 1-WL-bounded GNN produces the same
graph embedding for them — even though one is fully connected (compromise one
node, reach all) and the other splits into two isolated pieces. The distinguishing
property is **global** (how many connected pieces? how far can you reach?), and a
local aggregator cannot see it. That is the gap the rest of this repo exploits.

## 1.2 · A cover, and the matrix homomorphism `Tr`

Think of the adjacency matrix as a *bookkeeping device*: $A_{uv}=1$ records the
one-hop directed subgraph "the edge $u\to v$." The neighborhood of $v$ is just the
set of edges into $v$ — one particular family of subgraphs.

A **cover** generalizes this: it is a family of directed subgraphs (think
*directed paths*) that we attach to nodes. The framework gives a functor `Tr`
("trace") that

* sends a directed subgraph to its $0/1$ **matrix**, and — crucially —
* sends **subgraph composition** (gluing a path ending at $x$ to a path starting
  at $x$) to **matrix multiplication**.

That second property is a *monoid homomorphism*: $\mathrm{Tr}(D_1 \circ D_2) =
\mathrm{Tr}(D_1)\,\mathrm{Tr}(D_2)$. Matrix multiplication **is** path
concatenation. This is what lets us reason about long-range structure with linear
algebra. Let's verify it on a concrete random example.

In [3]:
from src.covers import Tr, compose, check_homomorphism

rng = np.random.default_rng(0)
D1 = (rng.random((4, 4)) < 0.4).astype(float)   # two small directed subgraphs
D2 = (rng.random((4, 4)) < 0.4).astype(float)

print("Tr(D1) =\n", Tr(D1))
print("\nsubgraph composition  Tr(D1 o D2) =\n", compose(D1, D2))
print("\nmatrix product (Tr D1)(Tr D2), booleanized =\n", ((Tr(D1) @ Tr(D2)) > 0).astype(float))
print("\nidentical? ", check_homomorphism(seed=0))

Tr(D1) =
 [[0. 1. 1. 1.]
 [0. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 1.]]

subgraph composition  Tr(D1 o D2) =
 [[1. 1. 0. 1.]
 [0. 0. 0. 0.]
 [0. 0. 0. 1.]
 [1. 1. 0. 1.]]

matrix product (Tr D1)(Tr D2), booleanized =
 [[1. 1. 0. 1.]
 [0. 0. 0. 0.]
 [0. 0. 0. 1.]
 [1. 1. 0. 1.]]

identical?  True


The composed subgraph and the (booleanized) matrix product agree entry for entry.
So once a cover is written in matrix form, **composing paths = multiplying
matrices**, and powers of a matrix enumerate longer and longer paths. Every cover
below is just a different way of organizing those powers.

## 1.3 · The walk cover

The simplest cover is the monoid **generated by the edge cover**: take the edge
subgraph $A$ and all its compositions. By the homomorphism, the $t$-fold
composition is exactly $A^t$, whose entries count **walks of length $t$**.

From the walk cover we read two per-node feature blocks, for each length $t$:

* **arriving walks** $A^t\mathbf{1}$ — how many length-$t$ walks land on the node,
* **closed walks** $\operatorname{diag}(A^t)$ — walks of length $t$ that *return*
  to the node (the cycle signal).

In [4]:
from src.covers import walk_cover

A = _flat(6)                       # directed 6-cycle
print("A (6-cycle):\n", A.astype(int))
print("\nA^2 (length-2 walks):\n", np.linalg.matrix_power(A, 2).astype(int))

# walk_cover stacks [arriving_t, closed_t] for t = 1..K as per-node columns
W = walk_cover(A, K=3)
print("\nwalk_cover columns = [arrive_1, closed_1, arrive_2, closed_2, arrive_3, closed_3]")
print("per-node walk features (one row per node):\n", W)
print("\nEvery node looks the same — a single cycle is vertex-transitive, so the")
print("walk cover alone is constant across nodes (this matters in notebook 03).")

A (6-cycle):
 [[0 1 0 0 0 0]
 [0 0 1 0 0 0]
 [0 0 0 1 0 0]
 [0 0 0 0 1 0]
 [0 0 0 0 0 1]
 [1 0 0 0 0 0]]

A^2 (length-2 walks):
 [[0 0 1 0 0 0]
 [0 0 0 1 0 0]
 [0 0 0 0 1 0]
 [0 0 0 0 0 1]
 [1 0 0 0 0 0]
 [0 1 0 0 0 0]]

walk_cover columns = [arrive_1, closed_1, arrive_2, closed_2, arrive_3, closed_3]
per-node walk features (one row per node):
 [[1. 0. 1. 0. 1. 0.]
 [1. 0. 1. 0. 1. 0.]
 [1. 0. 1. 0. 1. 0.]
 [1. 0. 1. 0. 1. 0.]
 [1. 0. 1. 0. 1. 0.]
 [1. 0. 1. 0. 1. 0.]]

Every node looks the same — a single cycle is vertex-transitive, so the
walk cover alone is constant across nodes (this matters in notebook 03).


## 1.4 · The reachability cover

Instead of *counting* walks at each length, the **reachability cover** takes the
union over all lengths: $\operatorname{sign}\!\big(\sum_t A^t\big)$. Its entry
$(u,v)$ is 1 iff $v$ is reachable from $u$. The per-node feature is the
**normalized blast radius** — the fraction of the estate a node can reach:

* $\approx 1.0$ in a *flat* estate (one cycle — every host reaches every host),
* $\approx 1/k$ in a $k$-segment estate (you only reach your own enclave).

This is the single scalar that solves demo `02`.

In [5]:
from src.covers import reachability_cover
from src.operators import blast_radius, n_components

for name, A in [("flat (1 cycle, n=12)", _flat(12)),
                ("segmented (k=3, n=12)", _segmented(12, 3))]:
    br = blast_radius(A)            # fraction reachable, including self
    print(f"{name:24s} blast radius per node ~ {br.mean():.3f}   "
          f"#components = {n_components(A)}")
print("\nThe flat estate's nodes each reach ~100% of hosts; the 3-segment estate's")
print("each reach ~1/3. That gap is invisible to 1-WL but trivial once exposed.")

flat (1 cycle, n=12)     blast radius per node ~ 1.000   #components = 1
segmented (k=3, n=12)    blast radius per node ~ 0.333   #components = 3

The flat estate's nodes each reach ~100% of hosts; the 3-segment estate's
each reach ~1/3. That gap is invisible to 1-WL but trivial once exposed.


## 1.5 · The sieve cover — where category theory earns its keep

The walk and reachability covers both look at paths **into** a node. They are
blind to a subtler thing: the structure **among the elements that cover a node**.

This is the **sieve**. In a Grothendieck *site*, a sieve on an object $v$ is a
family of morphisms into $v$ that is **closed under precomposition** — if
$g\colon x\to v$ is in the sieve and $h\colon w\to x$ is any morphism, then
$g\circ h\colon w\to v$ is in it too. Intuitively: "everything that maps into $v$,
together with how those things map among *themselves*."

The concrete instantiation we use: for each node $v$, look at the **subgraph
induced on $v$'s neighbors** — not just *that* they are neighbors, but how they
are wired to each other. Two graphs can be cospectral (identical walk/closed-walk
counts at every length) yet have different neighborhood structure — and the sieve
cover sees it.

In [6]:
from src.covers import sieve_cover
from src.srg_data import rook44, shrikhande

# Both are SRG(16,6,2,2): 6-regular, cospectral, 1-WL-indistinguishable (notebook 03).
# But each node's NEIGHBORHOOD induces a different subgraph.
for name, A in [("Rook 4x4", rook44()), ("Shrikhande", shrikhande())]:
    S = sieve_cover(A, L=6)
    tris = set(S[:, 0].astype(int))      # column 0 = #triangles in N(v)
    print(f"{name:12s}: triangles inside each node's neighborhood = {tris}")

print("\nRook neighborhoods are two disjoint triangles (2 triangles per node);")
print("Shrikhande neighborhoods are a single 6-cycle (0 triangles). Same C6-vs-2C3")
print("motif as section 1.1 — but now as a *local* invariant the sieve reads.")

Rook 4x4    : triangles inside each node's neighborhood = {2}
Shrikhande  : triangles inside each node's neighborhood = {0}

Rook neighborhoods are two disjoint triangles (2 triangles per node);
Shrikhande neighborhoods are a single 6-cycle (0 triangles). Same C6-vs-2C3
motif as section 1.1 — but now as a *local* invariant the sieve reads.


## 1.6 · Summary

| cover | matrix form | per-node feature | sees |
|---|---|---|---|
| **walk** | $A^t$ | arriving / closed walks | local walk counts |
| **reachability** | $\operatorname{sign}\sum_t A^t$ | normalized blast radius | global reachability |
| **sieve** | induced subgraph on $N(v)$ | neighborhood structure | arrangement, not just counts |

All three are *covers* in the same algebraic framework — designed by **choosing
which subgraphs to compose**, then translated to matrices by `Tr`. "Designing a
GNN" becomes "choosing a cover."

- **Notebook 02** — the reachability cover beats 1-WL on a blast-radius task
  (but a humble component-count ties it: the *signal* matters, not the model).
- **Notebook 03** — a cospectral pair where walk/reachability covers *provably*
  fail and only the **sieve cover** separates them.
- **Notebook 04** — a **temporal** cover where the sieve closure becomes the
  literal **causal past** of an event.
